In [ ]:
# Cell 1: Complete Self-Contained Visualization Setup

# 1. Import Sedona Context and the updated Maps module to avoid deprecation warnings
from sedona.spark import *
from sedona.spark.maps.SedonaKepler import SedonaKepler

print("Initializing Sedona Context...")
# 2. Build or retrieve the active Spark session managed by Wherobots
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

print("Extracting data samples for visualization...")
try:
    bbox = "POLYGON ((151.10 -33.15, 151.85 -33.15, 151.85 -32.70, 151.10 -32.70, 151.10 -33.15))"
    
    # 3. Query layers directly from your Iceberg catalog tables
    print("Loading Hunter Demographics...")
    demo_df = sedona.sql(f"""
        SELECT sa2_code, sa2_name_spatial, pop_estimate, pop_density, geometry 
        FROM org_catalog.fgsdb.abs_demographics 
        WHERE year = 2025 
          AND (sa3_name = 'Newcastle' OR sa3_name = 'Lake Macquarie' OR sa3_name = 'Hunter Valley' OR 
               ST_Contains(ST_GeomFromText('{bbox}'), geometry))
    """)
    
    print("Loading Rail Stations...")
    stations_df = sedona.sql(f"""
        SELECT name, station_class, geometry 
        FROM org_catalog.fgsdb.nsw_rail_stations 
        WHERE ST_Contains(ST_GeomFromText('{bbox}'), geometry)
    """)

    print("Loading Overture Road network segments...")
    roads_df = sedona.sql(f"""
        SELECT id, subtype, geometry 
        FROM wherobots_open_data.overture_maps_foundation.transportation_segment 
        WHERE ST_Contains(ST_GeomFromText('{bbox}'), geometry)
          AND subtype = 'road'
        LIMIT 1000
    """)

    print("Loading Opal Transit Patronage...")
    patronage_df = sedona.sql(f"""
        SELECT stop_name, travel_mode, SUM(trips) AS total_trips, geometry 
        FROM org_catalog.fgsdb.tfnsw_opal_usage 
        WHERE year = 2025 
          AND ST_Within(geometry, ST_GeomFromText('{bbox}')) 
        GROUP BY stop_name, travel_mode, geometry
    """)

    print("Running Service Loss Catchment Analysis...")
    loss_catchment_df = sedona.sql(f"""
        SELECT 
            p.stop_name, 
            p.travel_mode, 
            SUM(p.trips) AS total_trips, 
            p.geometry
        FROM 
            org_catalog.fgsdb.tfnsw_opal_usage p
        LEFT JOIN (
            SELECT DISTINCT p2.stop_name
            FROM org_catalog.fgsdb.tfnsw_opal_usage p2
            JOIN wherobots_open_data.overture_maps_foundation.transportation_segment r
            ON ST_Distance(p2.geometry, r.geometry) <= 0.018
            WHERE ST_Within(p2.geometry, ST_GeomFromText('{bbox}'))
              AND ST_Contains(ST_GeomFromText('{bbox}'), r.geometry)
              AND r.subtype = 'road'
        ) roads
        ON p.stop_name = roads.stop_name
        WHERE 
            p.year = 2025
            AND ST_Within(p.geometry, ST_GeomFromText('{bbox}'))
            AND roads.stop_name IS NULL
        GROUP BY 
            p.stop_name, p.travel_mode, p.geometry
    """)

    # 4. Initialize the Kepler.gl map with the demographic layer
    print("Generating map canvas centered on Newcastle / Lake Macquarie...")
    map_config = {
        "mapState": {
            "bearing": 0,
            "dragRotate": False,
            "latitude": -32.9283,
            "longitude": 151.7817,
            "pitch": 0,
            "zoom": 10,
            "isSplit": False
        }
    }
    map_vis = SedonaKepler.create_map(df=demo_df, name="Hunter Demographics (SA2)", config=map_config)

    # 5. Overlay all transport network and analysis layers
    print("Adding Rail Stations...")
    SedonaKepler.add_df(map_vis, df=stations_df, name="NSW Rail Stations")
    
    print("Adding Major Road Segments...")
    SedonaKepler.add_df(map_vis, df=roads_df, name="Hunter Road Network")
    
    print("Adding Opal Patronage...")
    SedonaKepler.add_df(map_vis, df=patronage_df, name="Opal Patronage Trips")
    
    print("Adding Service Loss Catchment...")
    SedonaKepler.add_df(map_vis, df=loss_catchment_df, name="Service Loss Catchment (Stops > 2km from Roads)")

    # 6. Export the interactive map as a standalone HTML file in your Jupyter workspace
    output_file = "raw_data_explorer.html"
    map_vis.save_to_html(file_name=output_file)

    print(f"\n🚀 Success! Map saved to your workspace as: {output_file}")
    print("Refresh your file explorer side-panel, download 'raw_data_explorer.html', and open it locally.")

except Exception as e:
    print(f"\n❌ Execution failed: {str(e)}")
    print("Please verify that the tables exist in 'org_catalog.fgsdb' and that your notebook runtime is active.")
